In [ ]:
# Setup
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """The qmulp checkout this notebook lives in."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the qmulp project root")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fmac.config import raw_csvs

DATA_PATHS = raw_csvs()
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
plt.style.use("default")
print("Repository root:", ROOT)
print("cohort files:", {year: path.name for year, path in DATA_PATHS.items()})

In [ ]:
# Load both raw datasets without modifying them
cohorts = {year: pd.read_csv(path) for year, path in DATA_PATHS.items()}
df_2023 = cohorts["2023"]
df_2024 = cohorts["2024"]
print({year: frame.shape for year, frame in cohorts.items()})

In [ ]:
# Source-file and shape overview
source_rows = []
for year, path in DATA_PATHS.items():
    frame = cohorts[year]
    source_rows.append(
        {
            "cohort": year,
            "path": str(path.relative_to(ROOT)),
            "file_size_bytes": path.stat().st_size,
            "rows": len(frame),
            "columns": frame.shape[1],
        }
    )
source_overview = pd.DataFrame(source_rows).set_index("cohort")
display(source_overview)
print("Same column names and order:", list(df_2023.columns) == list(df_2024.columns))
print("Only in 2023:", sorted(set(df_2023.columns) - set(df_2024.columns)))
print("Only in 2024:", sorted(set(df_2024.columns) - set(df_2023.columns)))

confirmed no columns that are uniquely in each dataset.

In [ ]:
# Unmodified sample rows
for year, frame in cohorts.items():
    print(f"\n{year} first five rows")
    display(frame.head())

In [ ]:
# Schema comparison
all_columns = list(dict.fromkeys([*df_2023.columns, *df_2024.columns]))
schema_rows = []
for column in all_columns:
    row = {"column": column}
    for year, frame in cohorts.items():
        if column in frame:
            row[f"dtype_{year}"] = str(frame[column].dtype)
            row[f"non_null_{year}"] = int(frame[column].notna().sum())
            row[f"unique_{year}"] = int(frame[column].nunique(dropna=False))
        else:
            row[f"dtype_{year}"] = "ABSENT"
            row[f"non_null_{year}"] = np.nan
            row[f"unique_{year}"] = np.nan
    row["dtype_agrees"] = row["dtype_2023"] == row["dtype_2024"]
    schema_rows.append(row)
schema_table = pd.DataFrame(schema_rows).set_index("column")
display(schema_table)

# nulls for 2023 


# nulls for 2024 



In [ ]:
# Numeric descriptive statistics
for year, frame in cohorts.items():
    print(f"\n{year} numeric summary")
    numeric_summary = frame.select_dtypes(include=np.number).describe().T
    numeric_summary["unique"] = frame.select_dtypes(include=np.number).nunique(dropna=False)
    display(numeric_summary)

In [ ]:
# Missing and null data
missing_rows = []
for year, frame in cohorts.items():
    for column in frame.columns:
        count = int(frame[column].isna().sum())
        missing_rows.append(
            {
                "cohort": year,
                "column": column,
                "null_count": count,
                "null_percent": 100 * count / len(frame),
            }
        )
missing_table = (
    pd.DataFrame(missing_rows)
    .sort_values(["null_percent", "cohort", "column"], ascending=[False, True, True])
    .reset_index(drop=True)
)
display(missing_table)

In [ ]:
# String-column inventory for later OHE decisions; no encoding recommendation is made
string_rows = []
for year, frame in cohorts.items():
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        string_rows.append(
            {
                "cohort": year,
                "column": column,
                "unique_including_null": int(frame[column].nunique(dropna=False)),
                "unique_to_row_ratio": frame[column].nunique(dropna=False) / len(frame),
                "null_count": int(frame[column].isna().sum()),
                "null_percent": 100 * frame[column].isna().mean(),
            }
        )
string_inventory = pd.DataFrame(string_rows).sort_values(
    ["cohort", "unique_including_null", "column"]
)
display(string_inventory)

In [ ]:
# Values and frequencies for every string column
for year, frame in cohorts.items():
    print(f"\n{'=' * 80}\n{year} STRING VALUES")
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        counts = frame[column].value_counts(dropna=False)
        frequency = pd.DataFrame(
            {
                "count": counts,
                "percent": 100 * counts / len(frame),
            }
        )
        print(f"\n{column}")
        display(frequency)

In [ ]:
# Blank, whitespace, and normalized string-collision checks
string_quality_rows = []
for year, frame in cohorts.items():
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        non_null = frame[column].dropna().astype(str)
        stripped = non_null.str.strip()
        normalized = stripped.str.casefold()
        raw_unique = non_null.nunique()
        normalized_unique = normalized.nunique()
        string_quality_rows.append(
            {
                "cohort": year,
                "column": column,
                "blank_strings": int(stripped.eq("").sum()),
                "leading_or_trailing_whitespace": int(non_null.ne(stripped).sum()),
                "raw_unique": int(raw_unique),
                "normalized_unique": int(normalized_unique),
                "normalized_collisions": int(raw_unique - normalized_unique),
            }
        )
string_quality = pd.DataFrame(string_quality_rows)
display(string_quality)

In [ ]:
# Duplicate and loan-identifier checks
identifier_rows = []
for year, frame in cohorts.items():
    identifier_rows.append(
        {
            "cohort": year,
            "duplicate_rows": int(frame.duplicated().sum()),
            "null_loan_identifiers": int(frame["LoanIdentifier"].isna().sum()),
            "duplicate_loan_identifiers": int(frame["LoanIdentifier"].duplicated().sum()),
        }
    )
duplicate_table = pd.DataFrame(identifier_rows).set_index("cohort")
display(duplicate_table)
shared_identifiers = set(df_2023["LoanIdentifier"].dropna()) & set(
    df_2024["LoanIdentifier"].dropna()
)
print("Loan identifiers shared across cohorts:", len(shared_identifiers))
print("Shared identifier sample:", sorted(shared_identifiers)[:10])

No duplicate rows for a loan ID found.

In [ ]:
# Constant, near-constant, and dominant numeric values for sentinel review
constant_rows = []
dominant_numeric_rows = []
for year, frame in cohorts.items():
    for column in frame.columns:
        counts = frame[column].value_counts(dropna=False)
        largest_count = int(counts.iloc[0])
        largest_share = largest_count / len(frame)
        if frame[column].nunique(dropna=False) == 1 or largest_share >= 0.95:
            constant_rows.append(
                {
                    "cohort": year,
                    "column": column,
                    "unique_including_null": int(frame[column].nunique(dropna=False)),
                    "dominant_value": counts.index[0],
                    "dominant_count": largest_count,
                    "dominant_percent": 100 * largest_share,
                }
            )
        if pd.api.types.is_numeric_dtype(frame[column]) and largest_share >= 0.95:
            dominant_numeric_rows.append(
                {
                    "cohort": year,
                    "column": column,
                    "dominant_value": counts.index[0],
                    "dominant_percent": 100 * largest_share,
                }
            )
constant_table = pd.DataFrame(constant_rows).sort_values(
    ["cohort", "dominant_percent"], ascending=[True, False]
)
dominant_numeric_table = pd.DataFrame(dominant_numeric_rows)
print("Constant and near-constant columns")
display(constant_table)
print("Dominant numeric values to review as legitimate constants, imbalance, or sentinels")
display(dominant_numeric_table)

In [ ]:
# Date parsing and loan-term consistency checks using temporary validation series
parsed_dates = {}
date_rows = []
for year, frame in cohorts.items():
    first_payment = pd.to_datetime(frame["FirstPaymentDate"], errors="coerce")
    maturity_text = frame["MaturityDate"].astype("Int64").astype("string").str.zfill(6)
    maturity = pd.to_datetime(maturity_text, format="%Y%m", errors="coerce")
    calculated_months = (
        (maturity.dt.year - first_payment.dt.year) * 12 + maturity.dt.month - first_payment.dt.month
    )
    term_difference = calculated_months - frame["OriginalLoanTerm"]
    parsed_dates[year] = {
        "first_payment": first_payment,
        "maturity": maturity,
        "term_difference": term_difference,
    }
    date_rows.append(
        {
            "cohort": year,
            "first_payment_parse_failures": int(first_payment.isna().sum()),
            "first_payment_min": first_payment.min(),
            "first_payment_max": first_payment.max(),
            "maturity_parse_failures": int(maturity.isna().sum()),
            "maturity_min": maturity.min(),
            "maturity_max": maturity.max(),
            "maturity_before_first_payment": int((maturity < first_payment).sum()),
            "term_difference_beyond_2_months": int(term_difference.abs().gt(2).sum()),
        }
    )
date_validation = pd.DataFrame(date_rows).set_index("cohort")
display(date_validation)
for year in cohorts:
    print(f"\n{year} calculated maturity-month difference minus OriginalLoanTerm")
    display(parsed_dates[year]["term_difference"].value_counts(dropna=False).sort_index())

In [ ]:
# Basic invalid-data flags; findings are review flags and are not removed or corrected
validation_results = []
validation_masks = {}
for year, frame in cohorts.items():
    masks = {
        "target_not_0_or_1": ~frame["target"].isin([0, 1]),
        "property_state_not_FL": frame["PropertyState"].ne("FL") | frame["PropertyState"].isna(),
        "nonpositive_original_upb": frame["OriginalUPB"].le(0) | frame["OriginalUPB"].isna(),
        "nonpositive_original_loan_term": frame["OriginalLoanTerm"].le(0)
        | frame["OriginalLoanTerm"].isna(),
        "nonpositive_number_of_borrowers": frame["NumberOfBorrowers"].le(0)
        | frame["NumberOfBorrowers"].isna(),
        "nonpositive_number_of_units": frame["NumberOfUnits"].le(0) | frame["NumberOfUnits"].isna(),
        "negative_mi_percent": frame["MI%"].lt(0),
        "negative_cltv": frame["CLTV"].lt(0),
        "negative_dti": frame["DTI"].lt(0),
        "negative_ltv": frame["LTV"].lt(0),
        "ltv_greater_than_cltv": frame["LTV"].gt(frame["CLTV"]),
        "fico_outside_300_to_850": ~frame["FICO"].between(300, 850),
        "nonpositive_interest_rate": frame["OriginalInterestRate"].le(0)
        | frame["OriginalInterestRate"].isna(),
        "first_payment_parse_failure": parsed_dates[year]["first_payment"].isna(),
        "maturity_parse_failure": parsed_dates[year]["maturity"].isna(),
        "maturity_before_first_payment": parsed_dates[year]["maturity"].lt(
            parsed_dates[year]["first_payment"]
        ),
        "term_difference_beyond_2_months": parsed_dates[year]["term_difference"].abs().gt(2),
    }
    validation_masks[year] = masks
    for check, mask in masks.items():
        validation_results.append(
            {"cohort": year, "check": check, "flagged_rows": int(mask.fillna(False).sum())}
        )
validation_table = pd.DataFrame(validation_results)
display(validation_table)

for year, masks in validation_masks.items():
    frame = cohorts[year]
    for check, mask in masks.items():
        mask = mask.fillna(False)
        if mask.any():
            print(f"\n{year} — {check} — sample")
            display(frame.loc[mask].head(5))

In [ ]:
# Numeric skewness, excluding target, date/geographic codes, and constant columns
skew_exclusions = {"target", "MaturityDate", "MSA", "PostalCode"}
skew_columns = [
    column
    for column in df_2023.select_dtypes(include=np.number).columns
    if column not in skew_exclusions
    and df_2023[column].nunique(dropna=True) > 1
    and df_2024[column].nunique(dropna=True) > 1
]
skew_table = pd.DataFrame(
    {year: frame[skew_columns].skew(numeric_only=True) for year, frame in cohorts.items()}
)
skew_table["maximum_absolute_skew"] = skew_table.abs().max(axis=1)
skew_table = skew_table.sort_values("maximum_absolute_skew", ascending=False)
display(skew_table)

In [ ]:
# Histograms for numeric columns included in the skew analysis
ncols = 3
nrows = int(np.ceil(len(skew_columns) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.atleast_1d(axes).ravel()
for axis, column in zip(axes, skew_columns):
    for year, frame in cohorts.items():
        axis.hist(frame[column].dropna(), bins=30, alpha=0.5, label=year)
    axis.set_title(column)
    axis.set_ylabel("Count")
    axis.legend()
for axis in axes[len(skew_columns) :]:
    axis.set_visible(False)
fig.suptitle("Numeric distributions by cohort", y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for numeric columns included in the skew analysis
ncols = 3
nrows = int(np.ceil(len(skew_columns) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.atleast_1d(axes).ravel()
for axis, column in zip(axes, skew_columns):
    values = [cohorts[year][column].dropna() for year in cohorts]
    axis.boxplot(values, tick_labels=list(cohorts))
    axis.set_title(column)
    axis.set_ylabel(column)
for axis in axes[len(skew_columns) :]:
    axis.set_visible(False)
fig.suptitle("Numeric boxplots by cohort", y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# Target counts and percentages
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
target_rows = []
for axis, (year, frame) in zip(axes, cohorts.items()):
    counts = frame["target"].value_counts(dropna=False).sort_index()
    for value, count in counts.items():
        target_rows.append(
            {
                "cohort": year,
                "target": value,
                "count": int(count),
                "percent": 100 * count / len(frame),
            }
        )
    counts.plot.bar(ax=axis, title=f"{year} target distribution")
    axis.set_xlabel("target")
    axis.set_ylabel("Count")
    axis.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
target_table = pd.DataFrame(target_rows)
display(target_table)

In [ ]:
# Compact 2023-versus-2024 numeric comparison
comparison_rows = []
for column in df_2023.select_dtypes(include=np.number).columns:
    comparison_rows.append(
        {
            "column": column,
            "mean_2023": df_2023[column].mean(),
            "mean_2024": df_2024[column].mean(),
            "median_2023": df_2023[column].median(),
            "median_2024": df_2024[column].median(),
            "missing_percent_2023": 100 * df_2023[column].isna().mean(),
            "missing_percent_2024": 100 * df_2024[column].isna().mean(),
        }
    )
cohort_comparison = pd.DataFrame(comparison_rows).set_index("column")
display(cohort_comparison)
print("Target rate 2023:", df_2023["target"].mean())
print("Target rate 2024:", df_2024["target"].mean())

In [ ]:
# Final review summary; no cleaning or encoding decisions are applied
schema_disagreements = schema_table.index[~schema_table["dtype_agrees"]].tolist()
missing_columns = missing_table.loc[missing_table["null_count"].gt(0), ["cohort", "column"]]
constant_columns = constant_table.loc[
    constant_table["unique_including_null"].eq(1), ["cohort", "column", "dominant_value"]
]
flagged_checks = validation_table.loc[validation_table["flagged_rows"].gt(0)]
string_columns = sorted(set(string_inventory["column"]))

print("Schema dtype disagreements:", schema_disagreements)
print("\nColumns containing null values")
display(missing_columns.drop_duplicates().reset_index(drop=True))
print("\nConstant columns")
display(constant_columns.reset_index(drop=True))
print("\nDominant numeric values requiring review")
display(dominant_numeric_table.reset_index(drop=True))
print("\nValidity checks with flagged rows")
display(flagged_checks.reset_index(drop=True))
print("\nString columns requiring a later OHE or exclusion decision")
print(string_columns)